# 목표

이 노트북은 Ray Train과 PyTorch를 사용하여 **Oxford-IIIT Pet 데이터셋**을 분산 학습하는 방법을 실습하기 위한 파일입니다. 아래 각 셀의 지시사항에 따라 빈 셀(Code)을 추가하여 코드를 작성하세요.

**[Step 1: 라이브러리 설치 및 환경 설정]**

- 환경 변수 `RAY_TRAIN_V2_ENABLED`를 `"1"`로 설정하세요.
- 필요한 패키지를 설치하는 명령어를 작성하세요:
  설치할 패키지: `ray[train]`, `torch`, `torchvision`, `matplotlib`, `tqdm`, `ipywidgets`

**[Step 2: 모듈 임포트]**

- 기본 파이썬 모듈: `os`, `tempfile`, `uuid`, `typing.Dict`
- PyTorch 핵심 및 비전 처리: `torch`, `torch.nn`, `torch.utils.data.DataLoader`, `torchvision.datasets`, `torchvision.transforms` (`Normalize`, `ToTensor`, `Resize`)
- 기타 도구: `filelock.FileLock`, `tqdm.auto.tqdm`
- Ray Train 관련: `ray.train`, `ray.train.RunConfig`, `ray.train.ScalingConfig`, `ray.train.torch.TorchTrainer`

**[Step 3: 데이터 로더 구현 (Oxford-IIIT Pet)]**

- `get_dataloaders(batch_size)` 함수를 작성하세요.
- 이미지 크기를 64x64로 통일하기 위해 `Resize` 변환을 포함하여 Transform을 구성하세요.
- 다중 워커 환경에서의 동시 다운로드 방지를 위해 `FileLock`을 사용하세요.
- `torchvision.datasets.OxfordIIITPet`을 사용하여 `trainval` 스플릿과 `test` 스플릿을 불러온 후, 각각의 DataLoader를 반환하세요.

**[Step 4: 모델 아키텍처 정의 (Custom ResNet)]**

- `torchvision.models`의 사전학습 모델을 사용하지 않고, 직접 `nn.Module`을 상속받는 `ResNet` 기반의 클래스를 작성하세요.
- 모델의 마지막 분류기(classifier) 출력 형태는 Oxford-IIIT Pet 데이터셋의 클래스 수인 37개에 맞춰지도록 구성하세요.

**[Step 5: 분산 학습 워커 함수 구현 (Ray Train)]**

- `config`를 인자로 받는 `train_func_per_worker(config)` 함수를 작성하세요.
- 앞서 작성한 `get_dataloaders`를 호출하여 데이터로더를 가져온 뒤, `ray.train.torch.prepare_data_loader`를 사용하여 분산 학습용으로 래핑하세요.
- Oxford-IIIT Pet 데이터셋에 맞게 37개의 클래스를 가지는 위에서 작성한 커스텀 ResNet 모델을 선언하고, `ray.train.torch.prepare_model`을 사용하여 분산 모델로 래핑하세요.
- 교차 엔트로피 손실 함수(CrossEntropyLoss)와 AdamW 옵티마이저를 설정하세요.
- 매 에폭(epoch)마다 모델을 학습시키고, Validation을 수행하세요. (여러 GPU를 사용할 경우 훈련 시작 전에 샘플러의 에폭을 수동으로 갱신해야 할 수 있습니다.)
- 에폭 종료 시 임시 디렉토리에 체크포인트를 저장하고, `ray.train.report`를 통해 손실(loss), 정확도(accuracy) 메트릭과 함께 체크포인트를 보고하세요.

**[Step 6: 메인 실행부 구성 및 학습 시작 (TorchTrainer)]**

- 학습률(`lr`), 총 에폭(`epochs`), 워커 당 배치 사이즈(`batch_size_per_worker`)가 포함된 학습 설정 딕셔너리(`train_config`)를 정의하세요.
- 워커의 개수와 GPU 사용 여부를 결정하는 `ScalingConfig`를 생성하세요.
- 고유한 훈련 이름을 포함하는 `RunConfig`를 설정하세요.
- `TorchTrainer` 인스턴스를 생성하고 위에서 정의한 설정과 학습 함수를 주입하세요.
- `ray.init()`를 호출하여 임시 Ray 클러스터를 생성하여 활용합니다.
- 메인 실행 함수(예: `train_oxford_pet`)를 직접 호출하여 분산 학습을 시작하고 결과를 출력하는 코드를 작성하세요.

**[Step 7: 결과 시각화]**

- `matplotlib.pyplot` 라이브러리를 임포트하세요.
- 체크포인트에 저장된 학습 모델을 로드합니다.
- 테스트 데이터셋(Test data) 중 일부 이미지를 추출하여 예측(prediction)을 수행하세요.
- 원본 이미지와 함께 예측된 라벨(Predicted Label) 및 실제 라벨(Actual Label)을 비교할 수 있도록 시각화(plot) 코드를 작성하세요.